# ACE-Step UI - Google Colab 部署

一键部署 ACE-Step AI 音乐生成器到 Google Colab。

**架构**：
- ACE-Step API (Python/Gradio, port 8001) — AI 推理引擎
- Express Backend (Node.js, port 3001) — REST API + 静态文件托管
- React Frontend (构建后由 Express 托管) — 用户界面

**运行前请确认**：
1. 已选择 GPU 运行时 (Runtime → Change runtime type → **T4 GPU**)
2. Colab 环境正常运行

> **重要**：ACE-Step 依赖 CUDA 进行模型推理，**必须使用 GPU 运行时**。TPU 运行时**不兼容**。

## 1. 检测运行环境

In [ ]:
#@title 检测 GPU / TPU / CPU 环境
import torch
import os

RUNTIME_TYPE = "unknown"
DEVICE = "cpu"

# ── 检测 GPU (CUDA) ──────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    RUNTIME_TYPE = "gpu"
    DEVICE = "cuda"
    print(f"[OK] GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"     CUDA version: {torch.version.cuda}")
    print(f"     PyTorch: {torch.__version__}")
    if gpu_mem < 4:
        print("     WARNING: GPU has <4GB VRAM. Generation may fail or be very slow.")

# ── 检测 TPU ─────────────────────────────────────────────
elif os.environ.get("COLAB_TPU_ADDR"):
    RUNTIME_TYPE = "tpu"
    tpu_addr = os.environ["COLAB_TPU_ADDR"]
    print("=" * 60)
    print("  [ERROR] TPU runtime detected — ACE-Step is NOT compatible")
    print("=" * 60)
    print()
    print(f"  TPU address: {tpu_addr}")
    print()
    print("  ACE-Step requires CUDA (NVIDIA GPU) for model inference.")
    print("  TPU uses a different compute model (torch_xla) that is not")
    print("  supported by ACE-Step's architecture.")
    print()
    print("  Fix: Runtime → Change runtime type → T4 GPU → Save")
    print("  Then restart this notebook from Cell 1.")
    print("=" * 60)

# ── CPU fallback ─────────────────────────────────────────
else:
    RUNTIME_TYPE = "cpu"
    DEVICE = "cpu"
    print("[WARNING] No GPU or TPU detected. Running on CPU.")
    print("  Music generation will be extremely slow (10-30 min per song).")
    print("  Fix: Runtime → Change runtime type → T4 GPU → Save")

# Save to env for later cells
os.environ["ACESTEP_DEVICE"] = DEVICE
os.environ["RUNTIME_TYPE"] = RUNTIME_TYPE
print(f"\nRuntime: {RUNTIME_TYPE.upper()}, Device: {DEVICE}")

## 2. 安装 Node.js

In [ ]:
#@title 安装 Node.js 20.x
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs

import subprocess
node_v = subprocess.getoutput('node --version')
npm_v = subprocess.getoutput('npm --version')
print(f"Node.js: {node_v}, npm: {npm_v}")

## 3. 克隆项目仓库

In [ ]:
#@title 克隆 ACE-Step 和 UI 仓库
#@markdown 修改下方 URL 为你的仓库地址（如使用 fork）

ACESTEP_REPO = "https://github.com/ace-step/ACE-Step.git"  #@param {type:"string"}
UI_REPO = "https://github.com/GitKK-work/ace-step-ui.git"  #@param {type:"string"}

!git clone --depth 1 "$ACESTEP_REPO" /content/ACE-Step-1.5
!git clone --depth 1 "$UI_REPO" /content/ace-step-ui

print("\nRepositories cloned:")
!ls -la /content/ACE-Step-1.5/
!ls -la /content/ace-step-ui/

## 4. 安装 Python 依赖 (ACE-Step)

In [ ]:
#@title 安装 ACE-Step Python 包
%cd /content/ACE-Step-1.5

# Install ACE-Step dependencies (uses setup.py + requirements.txt)
!pip install -e . 2>&1 | tail -10

# Verify
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Check acestep CLI is installed
import shutil
if shutil.which('acestep'):
    print(f'acestep CLI: {shutil.which("acestep")}')
else:
    print('WARNING: acestep CLI not found after install!')


## 5. 安装 Node.js 依赖

In [ ]:
#@title 安装前端和后端 npm 依赖

# Install frontend dependencies
%cd /content/ace-step-ui
!npm install 2>&1 | tail -5

# Install backend dependencies
%cd /content/ace-step-ui/server
!npm install 2>&1 | tail -5

print("\nNode.js dependencies installed.")

## 6. 配置环境变量

In [ ]:
#@title 生成 .env 配置文件
%cd /content/ace-step-ui

import os

env_content = """
PORT=3001
NODE_ENV=production
DATABASE_PATH=/content/ace-step-ui/data/acestep.db
ACESTEP_API_URL=http://localhost:8001
AUDIO_DIR=/content/ace-step-ui/public/audio
STATIC_DIR=/content/ace-step-ui/dist
FRONTEND_URL=http://localhost:3001
DATASETS_DIR=/content/ACE-Step-1.5/datasets
DATASETS_UPLOADS_DIR=/content/ACE-Step-1.5/datasets/uploads
JWT_SECRET=ace-step-colab-secret-change-me
""".strip()

with open('.env', 'w') as f:
    f.write(env_content + '\n')

print(".env file created:")
!cat .env

## 7. 构建前端

In [ ]:
#@title 构建 React 前端静态文件
%cd /content/ace-step-ui
!npm run build 2>&1 | tail -15

if os.path.exists('dist'):
    file_count = sum(len(files) for _, _, files in os.walk('dist'))
    print(f"\nFrontend built successfully: {file_count} files in dist/")
else:
    print("ERROR: Build failed! Check the output above.")

## 8. 创建必要目录

In [ ]:
#@title 创建数据和音频目录
!mkdir -p /content/ace-step-ui/data
!mkdir -p /content/ace-step-ui/public/audio
!mkdir -p /content/ACE-Step-1.5/datasets/uploads

print("Directories created.")

## 9. 启动 ACE-Step API 服务

**仅在 GPU 运行时下可用。** 如果你看到 Cell 1 报告 TPU，请先切换到 GPU 运行时。

In [ ]:
#@title 启动 ACE-Step 推理服务 (port 8001)
import subprocess, time, os, sys, shutil

# -- 运行时检查 --
runtime = os.environ.get('RUNTIME_TYPE', 'unknown')
if runtime == 'tpu':
    print('[BLOCKED] Cannot start ACE-Step on TPU runtime.')
    print('Fix: Runtime -> Change runtime type -> T4 GPU -> Save')
    print('Then restart from Cell 1.')
    sys.exit(1)

if runtime == 'unknown':
    print('[WARNING] Runtime not detected. Cell 1 may not have been run.')
    print('Continuing anyway, but ACE-Step may fail if no GPU is available.')
    print()

%cd /content/ACE-Step-1.5

# -- 确认 acestep CLI 可用 --
acestep_bin = shutil.which('acestep')
if not acestep_bin:
    print('[ERROR] acestep CLI not found. The Python package may not be installed.')
    print('Run Cell 4 first to install ACE-Step.')
    sys.exit(1)
print(f'Found acestep CLI: {acestep_bin}')

# -- 启动 ACE-Step Gradio 服务 --
LOG_FILE = '/content/acestep-api.log'

# acestep CLI uses click options: --port, --server_name, --share, --bf16

# Kill any existing process on port 8001
!fuser -k 8001/tcp 2>/dev/null; sleep 1

cmd = ['acestep', '--port', '8001', '--server_name', '127.0.0.1']
cmd_str = ' '.join(cmd)
print(f'Starting: {cmd_str}')

log_handle = open(LOG_FILE, 'w')
acestep_proc = subprocess.Popen(
    cmd,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    cwd='/content/ACE-Step-1.5'
)

# Check if it crashes immediately
time.sleep(5)
if acestep_proc.poll() is not None:
    exit_code = acestep_proc.returncode
    print(f'[ERROR] Process exited immediately (code {exit_code}).')
    print('Log output:')
    !cat {LOG_FILE} | tail -30
else:
    pid = acestep_proc.pid
    print(f'Process started (PID: {pid})')
    print('Waiting for model to load (this may take 1-5 minutes)...')

    import urllib.request
    for i in range(120):
        time.sleep(5)
        # Check if process is still alive
        if acestep_proc.poll() is not None:
            exit_code = acestep_proc.returncode
            print(f'[ERROR] ACE-Step process died (exit code {exit_code}).')
            print('Last 30 lines of log:')
            !tail -30 {LOG_FILE}
            break
        try:
            resp = urllib.request.urlopen('http://127.0.0.1:8001/', timeout=3)
            elapsed = (i+1)*5
            print(f'ACE-Step API is ready! (took {elapsed}s)')
            break
        except urllib.error.URLError:
            if (i+1) % 4 == 0:
                elapsed = (i+1)*5
                print(f'  Still loading... ({elapsed}s)')
        except Exception as e:
            if (i+1) % 4 == 0:
                elapsed = (i+1)*5
                print(f'  Waiting... ({elapsed}s) - {type(e).__name__}: {e}')
    else:
        print('[ERROR] API did not respond within 10 minutes.')
        print('Last 30 lines of log:')
        !tail -30 {LOG_FILE}
        print('Possible causes:')
        print('  1. Model download in progress (check log above)')
        print('  2. Insufficient GPU memory (need 4GB+ VRAM)')
        print('  3. CUDA driver mismatch')


## 10. 启动 Express 后端服务

In [ ]:
#@title 启动 Express 后端 (port 3001)
%cd /content/ace-step-ui

import subprocess, time, json, os

# Kill any existing process on port 3001
!fuser -k 3001/tcp 2>/dev/null; sleep 1

backend_proc = subprocess.Popen(
    ['npx', 'tsx', 'server/src/index.ts'],
    env={**os.environ, 'NODE_ENV': 'production'},
    stdout=open('/content/express-backend.log', 'w'),
    stderr=subprocess.STDOUT
)

print(f'Express backend starting (PID: {backend_proc.pid})...')
time.sleep(5)

# Check process is alive
if backend_proc.poll() is not None:
    exit_code = backend_proc.returncode
    print(f'[ERROR] Backend process exited (code {exit_code}).')
    print('Log output:')
    !cat /content/express-backend.log | tail -20
else:
    # Verify backend is running
    import urllib.request
    try:
        resp = urllib.request.urlopen('http://127.0.0.1:3001/health', timeout=5)
        data = json.loads(resp.read())
        print(f'Backend is running: {data}')
    except urllib.error.URLError as e:
        print(f'[WARNING] Backend not responding: {e}')
        print('It may still be starting. Wait a moment and run the health check cell.')
    except Exception as e:
        print(f'[WARNING] Unexpected error: {type(e).__name__}: {e}')
        print('Check logs: !cat /content/express-backend.log')


## 11. 健康检查

In [ ]:
#@title 验证所有服务
import urllib.request, json

services = [
    ("ACE-Step API", "http://127.0.0.1:8001/"),
    ("Express Backend", "http://127.0.0.1:3001/health"),
]

all_ok = True
for name, url in services:
    try:
        resp = urllib.request.urlopen(url, timeout=5)
        print(f"  [OK] {name} (HTTP {resp.status})")
    except urllib.error.URLError as e:
        print(f"  [FAIL] {name}: Connection refused — service is not running")
        all_ok = False
    except Exception as e:
        print(f"  [FAIL] {name}: {type(e).__name__}: {e}")
        all_ok = False

if all_ok:
    print("\nAll services are healthy!")
else:
    print("\nSome services failed.")
    print("Troubleshooting:")
    print("  - ACE-Step failed → Check Cell 9 output and logs: !cat /content/acestep-api.log")
    print("  - Express failed  → Check Cell 10 output and logs: !cat /content/express-backend.log")
    print("  - TPU runtime?    → Switch to GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
#@title 诊断：检查前端构建和 Express 托管状态
import urllib.request, os

print('=== 1. Check dist/ directory ===')
dist_dir = '/content/ace-step-ui/dist'
if os.path.exists(dist_dir):
    files = []
    for root, dirs, fnames in os.walk(dist_dir):
        for f in fnames:
            rel = os.path.relpath(os.path.join(root, f), dist_dir)
            size = os.path.getsize(os.path.join(root, f))
            files.append((rel, size))
    count = len(files)
    print(f'  dist/ exists: {count} files')
    for f, s in sorted(files):
        print(f'    {f} ({s:,} bytes)')
else:
    print('  [ERROR] dist/ directory does NOT exist!')
    print('  Fix: re-run Cell 7 (npm run build)')

print('')
print('=== 2. Check Express log ===')
log_file = '/content/express-backend.log'
if os.path.exists(log_file):
    with open(log_file) as f:
        lines = f.readlines()
    relevant = [l.strip() for l in lines if 'Static' in l or 'production' in l or 'running' in l.lower()]
    if relevant:
        for l in relevant:
            print(f'  {l}')
    else:
        print('  No Static lines found. Last 10 lines:')
        for l in lines[-10:]:
            print(f'    {l.strip()}')
else:
    print('  [WARNING] express-backend.log not found')

print('')
print('=== 3. Test Express root response ===')
try:
    resp = urllib.request.urlopen('http://127.0.0.1:3001/', timeout=5)
    body = resp.read().decode('utf-8', errors='replace')
    clen = len(body)
    print(f'  GET / => HTTP {resp.status}, Content-Length: {clen}')
    print('  First 500 chars:')
    print(f'    {body[:500]}')
except urllib.error.URLError as e:
    print(f'  [FAIL] Cannot connect: {e}')
except Exception as e:
    print(f'  [ERROR] {type(e).__name__}: {e}')

print('')
print('=== 4. Test /health endpoint ===')
try:
    resp = urllib.request.urlopen('http://127.0.0.1:3001/health', timeout=5)
    data = resp.read().decode()
    print(f'  GET /health => {data}')
except Exception as e:
    print(f'  [FAIL] {e}')


## 12. 设置公网访问

Colab 运行在远程 VM，需要端口转发才能从浏览器访问。提供三种方式：

In [ ]:
#@title 方式 A: 使用 ngrok（推荐，完整 React UI）
#@markdown 暴露 Express 后端 (含完整 React UI) 到公网
#@markdown 需要免费 ngrok authtoken: https://dashboard.ngrok.com/signup

NGROK_AUTHTOKEN = ""  #@param {type:"string"}

if not NGROK_AUTHTOKEN:
    print("Please enter your ngrok authtoken.")
    print("Get a free token at: https://dashboard.ngrok.com/signup")
else:
    # Install ngrok
    !pip install pyngrok -q

    from pyngrok import ngrok

    # Set authtoken
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

    # Open tunnel to Express backend (port 3001)
    public_url = ngrok.connect(3001).public_url
    print(f"\nPublic URL (ACE-Step UI): {public_url}")
    print("Open this URL in your browser to use ACE-Step UI.")
    print("\nNOTE: First visit may take a few seconds to load.")

In [ ]:
#@title 方式 B: 使用 Gradio Share（无需注册，仅 Gradio UI）
#@markdown 此方式需在步骤 9 重启 ACE-Step 时添加 --share 参数
#@markdown 只暴露 Gradio 原生 UI，不含 ace-step-ui 的 React 界面

print("Gradio Share 方式需要在步骤 9 启动 ACE-Step 时添加 --share 参数")
print("修改步骤 9 的启动命令为:")
print('  python -m acestep --port 8001 --enable-api --server_name 0.0.0.0 --share')
print("\n这样会得到类似 https://xxxxx.gradio.live 的公网链接")
print("注意: 此链接仅暴露 Gradio 原生 UI，不含 ace-step-ui 的 React 界面")

In [ ]:
#@title 方式 C: 使用 localtunnel（无需注册，备选方案）
!npm install -g localtunnel 2>&1 | tail -3

import subprocess, time

lt_proc = subprocess.Popen(
    ["lt", "--port", "3001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Read output until we get the URL
for line in lt_proc.stdout:
    print(line.strip())
    if "https://" in line:
        break

print("\nOpen the URL above in your browser.")
print("NOTE: localtunnel may show a password page - click 'Click to Continue'.")

## 常用操作

In [ ]:
#@title 查看 ACE-Step API 日志
!tail -50 /content/acestep-api.log

In [ ]:
#@title 查看 Express 后端日志
!tail -50 /content/express-backend.log

In [ ]:
#@title 停止所有服务
import subprocess

for proc_name in ["acestep", "tsx", "node", "lt"]:
    result = subprocess.run(["pkill", "-f", proc_name], capture_output=True)
    if result.returncode == 0:
        print(f"Stopped {proc_name} processes.")

try:
    from pyngrok import ngrok
    ngrok.kill()
    print("Ngrok tunnels closed.")
except:
    pass

print("\nAll services stopped.")

In [ ]:
#@title 挂载 Google Drive（持久化数据，可选）
#@markdown 将数据库和音频文件存储到 Google Drive，避免断线丢失

MOUNT_DRIVE = False  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil

    # Create persistent directories
    !mkdir -p /content/drive/MyDrive/ace-step-data/audio
    !mkdir -p /content/drive/MyDrive/ace-step-data/db

    # Symlink data directories to Drive
    for src, dst in [
        ('/content/drive/MyDrive/ace-step-data/audio', '/content/ace-step-ui/public/audio'),
        ('/content/drive/MyDrive/ace-step-data/db', '/content/ace-step-ui/data'),
    ]:
        if os.path.exists(dst) and os.path.islink(dst):
            os.unlink(dst)
        elif os.path.exists(dst):
            shutil.rmtree(dst)
        os.symlink(src, dst)
        print(f"Linked {dst} → {src}")

    print("\nGoogle Drive mounted. Data will persist across sessions.")
else:
    print("Drive not mounted. Data will be lost when Colab disconnects.")